# BM25 Baseline + Preparation for Cross-Encoder

Ce notebook :
1. Charge 10% du corpus MS MARCO (seed=42)
2. Exécute BM25 baseline
3. Sauvegarde les résultats BM25 + corpus indexé pour le Cross-Encoder

In [4]:
# Setup
import sys
import random
import numpy as np
import pickle
from pathlib import Path
import torch

SEED = 42
SUBSET_PCT = 0.10

random.seed(SEED)
np.random.seed(SEED)

def find_root():
    """Repository root (the notebook lives in notebooks/)."""
    here = Path.cwd()
    return here.parent if here.name == "notebooks" else here

PROJECT_ROOT = find_root()
sys.path.insert(0, str(PROJECT_ROOT))

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('CPU mode')

GPU: Tesla P100-PCIE-16GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


In [5]:
# Imports
import yaml
from src.data.dataset_factory import get_dataset
from src.evaluation.metrics import evaluate_run
from src.evaluation.results_io import save_run
from src.retrieval.bm25.bm25_retriever import BM25Retriever

print('OK: Imports')

OK: Imports


In [6]:
# Config
with open(PROJECT_ROOT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

dataset_name = cfg['dataset']['name']
split = cfg['dataset']['split']
data_dir = cfg['dataset']['data_dir']
top_k = cfg['retrieval']['top_k']

print(f'Config: {dataset_name}, split={split}, top_k={top_k}')

Config: msmarco, split=dev, top_k=100


In [7]:
# Chargement des données - 10% subset
dataset = get_dataset(dataset_name, data_dir=str(PROJECT_ROOT / data_dir))
full_corpus, queries, qrels = dataset.get_split(split)

queries = {qid: q for qid, q in queries.items() if qid in qrels}

# Garde seulement 10% du corpus avec seed=42
# MAIS: inclure les passages pertinents pour chaque query!
all_pids = list(full_corpus.keys())
random.shuffle(all_pids)

# Trouver les passages pertinents pour chaque query
relevant_pids = set()
for qid, relevant in qrels.items():
    relevant_pids.update(relevant.keys())

print(f'Relevant passages: {len(relevant_pids)}')

# Ajouter les passages pertinents d'abord, puis completer avec 10%
subset_pids = list(relevant_pids)
n_remaining = int(len(all_pids) * SUBSET_PCT) - len(subset_pids)

if n_remaining > 0:
    other_pids = [pid for pid in all_pids if pid not in relevant_pids]
    random.shuffle(other_pids)
    subset_pids.extend(other_pids[:n_remaining])

corpus = {pid: full_corpus[pid] for pid in subset_pids}

print(f'Full corpus: {len(full_corpus):,}')
print(f'Subset ({SUBSET_PCT*100:.0f}%): {len(corpus):,}')
print(f'Relevant in subset: {sum(1 for pid in subset_pids if pid in relevant_pids)}')
print(f'Queries: {len(queries):,}')

MS MARCO files not found — downloading...


Extracting corpus...


Converting corpus: 8841823it [01:11, 124289.73it/s]


## BM25 Baseline

In [8]:
# BM25 Retrieval
bm25_index_path = PROJECT_ROOT / 'data' / dataset_name / f'bm25_index_seed{SEED}_subset{SUBSET_PCT}.pkl'

retriever = BM25Retriever(index_path=str(bm25_index_path))
retriever.index(corpus)
print('OK: BM25 indexed')

Construction de l'index BM25 sur 884,182 passages...


Tokénisation: 100%|██████████| 884182/884182 [00:24<00:00, 36648.71it/s]


In [9]:
# Retrieve
results = retriever.retrieve(queries, top_k=top_k)
print(f'OK: Retrieved {len(results)} queries')

Recherche BM25: 100%|██████████| 6980/6980 [2:54:18<00:00,  1.50s/it]

OK: Retrieved 6980 queries


In [10]:
# Evaluate
metrics = evaluate_run(qrels, results, ks=[10, 50, 100])
print('\n=== BM25 Metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')


=== BM25 Metrics ===
  MRR@10: 0.0001
  nDCG@10: 0.0002
  Recall@10: 0.0006
  Recall@50: 0.0014
  Recall@100: 0.0021


## Sauvegarde pour Cross-Encoder

In [11]:
# Sauvegarder everything needed for Cross-Encoder
cache_dir = PROJECT_ROOT / 'data' / dataset_name / 'cross_encoder_cache'
cache_dir.mkdir(parents=True, exist_ok=True)

# 1. BM25 results
with open(cache_dir / 'bm25_results.pkl', 'wb') as f:
    pickle.dump(results, f)

# 2. Corpus (pour检索)
with open(cache_dir / 'corpus.pkl', 'wb') as f:
    pickle.dump(corpus, f)

# 3. Queries
with open(cache_dir / 'queries.pkl', 'wb') as f:
    pickle.dump(queries, f)

# 4. Qrels
with open(cache_dir / 'qrels.pkl', 'wb') as f:
    pickle.dump(qrels, f)

# 5. Config
config = {
    'seed': SEED,
    'subset_pct': SUBSET_PCT,
    'n_corpus': len(corpus),
    'n_queries': len(queries),
    'metrics': metrics,
}
with open(cache_dir / 'config.pkl', 'wb') as f:
    pickle.dump(config, f)

print(f'OK: Saved to {cache_dir}')
print(f'  - bm25_results.pkl')
print(f'  - corpus.pkl')
print(f'  - queries.pkl')
print(f'  - qrels.pkl')
print(f'  - config.pkl')